# Export lightweight real-world inputs for Zenodo

This notebook creates the analysis-ready gridded inputs required to run TraCE-ST for the three real-world manuscript cases. Processing is delegated to the corresponding run scripts so variable order, standardization, regridding, and coordinate conventions remain consistent with the paper. The exports contain only `time × var × lat × lon` data and compact metadata; raw source products, trajectory ensembles, figures, and independent validation data such as IBTrACS are intentionally excluded.

In [3]:
from __future__ import annotations

import hashlib
import importlib.util
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

OUTPUT_DIR = Path("/glade/derecho/scratch/jhayron/DataCaStLeBTs/Zenodo/real_world_inputs/")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Override these defaults with environment variables when raw inputs live elsewhere.
DEBBY_INPUT_DIR = Path(os.environ.get("TRACE_ST_DEBBY_INPUT_DIR", "/glade/derecho/scratch/jhayron/DataCaStLeBTs/FilesDebby"))
PINATUBO_INPUT_DIR = Path(os.environ.get("TRACE_ST_PINATUBO_INPUT_DIR", "/glade/derecho/scratch/jhayron/DataCaStLeBTs/Sandia"))
PNW_INPUT_DIR = Path(os.environ.get("TRACE_ST_PNW_INPUT_DIR", "/glade/derecho/scratch/jhayron/DataCaStLeBTs/JointFiles"))

## Minimal analysis domains

Temporal cuts include every tracing step plus the longest local causal-discovery window explored in the manuscript. Spatial cuts retain the pathway domains used in the paper, with global longitude preserved for Pinatubo's circumglobal transport. All longitudes are stored on `[0, 360)` and values are written as compressed `float32`.

In [4]:
def load_script(filename, module_name):
    script_path = Path(filename)
    if not script_path.exists():
        script_path = Path.cwd() / "Scripts_Paper" / filename
    if not script_path.exists():
        raise FileNotFoundError(f"Cannot locate manuscript script: {filename}")
    spec = importlib.util.spec_from_file_location(module_name, script_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def subset_lon_0360(da, west, east):
    west, east = west % 360.0, east % 360.0
    if west <= east:
        return da.sel(lon=slice(west, east))
    return xr.concat(
        [da.sel(lon=slice(west, float(da.lon.max()))), da.sel(lon=slice(0.0, east))],
        dim="lon",
    )


def prepare_export(da, *, time_start, time_end, lat_bounds, lon_bounds, attrs):
    rename = {}
    if "latitude" in da.dims:
        rename["latitude"] = "lat"
    if "longitude" in da.dims:
        rename["longitude"] = "lon"
    if rename:
        da = da.rename(rename)
    da = da.assign_coords(lon=(da.lon % 360.0)).sortby(["time", "lat", "lon"])
    da = da.sel(time=slice(time_start, time_end), lat=slice(*lat_bounds))
    da = subset_lon_0360(da, *lon_bounds).transpose("time", "var", "lat", "lon")
    da = da.astype("float32").rename("trace_st_data")
    da.attrs.update(attrs)
    return da


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

In [5]:
debby = load_script("5_DebbyRuns.py", "trace_st_debby_export")
pinatubo = load_script("6_PinatuboRuns.py", "trace_st_pinatubo_export")
pnw = load_script("8_HW_PNW21_Runs_v2.py", "trace_st_pnw_export")

debby_data = prepare_export(
    debby.load_debby_data(DEBBY_INPUT_DIR),
    time_start="2006-08-10 19:00:00", time_end=debby.DATE_END_DEBBY,
    lat_bounds=(0.0, 40.0), lon_bounds=(-50.0, 40.0),
    attrs=dict(case="Tropical Storm Debby (2006)", target_variable="precip",
               target_time=debby.DATE_END_DEBBY, target_lat=debby.EVENT_LAT_DEBBY,
               target_lon_0360=debby.EVENT_LON_DEBBY % 360.0, timeres="1h",
               source_script="5_DebbyRuns.py"),
)

pinatubo_data = prepare_export(
    pinatubo.load_pinatubo_data(PINATUBO_INPUT_DIR),
    time_start="1991-06-13", time_end=pinatubo.DATE_END_PINATUBO,
    lat_bounds=(-50.0, 80.0), lon_bounds=(0.0, 359.999),
    attrs=dict(case="Mount Pinatubo eruption (1991)", target_variable="AEROD_v",
               target_time=pinatubo.DATE_END_PINATUBO, target_lat=pinatubo.EVENT_LAT_PINATUBO,
               target_lon_0360=pinatubo.EVENT_LON_PINATUBO % 360.0, timeres="1d",
               standardization="grid-cell 1991 mean and standard deviation before temporal cut",
               source_script="6_PinatuboRuns.py"),
)

pnw.PATH_FILES = str(PNW_INPUT_DIR)
pnw_raw = pnw.regrid_to_2p5(pnw.load_full_data_for_vars(pnw.VAR_COMBOS[0]))
pnw_data = prepare_export(
    pnw_raw,
    time_start="2021-05-27", time_end=pnw.DATE_END,
    lat_bounds=(10.0, 80.0), lon_bounds=(90.0, 320.0),
    attrs=dict(case="Pacific Northwest heatwave (2021)", target_variable="Z500",
               target_time=pnw.DATE_END, target_lat=pnw.EVENT_LAT, target_lon_0360=pnw.EVENT_LON,
               timeres="1d", source_script="8_HW_PNW21_Runs_v2.py"),
)

for name, data in {"debby": debby_data, "pinatubo": pinatubo_data, "pnw21": pnw_data}.items():
    print(name, dict(data.sizes), list(map(str, data["var"].values)), str(data.time.min().values), str(data.time.max().values))

debby {'time': 361, 'var': 3, 'lat': 161, 'lon': 361} ['Tb', 'vo', 'precip'] 2006-08-11T00:00:00.000000000 2006-08-26T00:00:00.000000000
pinatubo {'time': 20, 'var': 4, 'lat': 33, 'lon': 90} ['TMSO201', 'BURDENSO401', 'TMH2SO401', 'AEROD_v'] 1991-06-13T00:00:00.000000000 1991-07-02T00:00:00.000000000
pnw21 {'time': 35, 'var': 6, 'lat': 36, 'lon': 116} ['Z500', 'Z10', 'MTNLWRF', 'TCWV', 'MSLHF', 'MSSHF'] 2021-05-27T00:00:00.000000000 2021-06-30T00:00:00.000000000


In [9]:
exports = {"debby": debby_data, "pinatubo": pinatubo_data, "pnw21": pnw_data}
manifest = {"format": "analysis-ready TraCE-ST inputs", "longitude_convention": "0 to 360 degrees east", "files": {}}

for case_name, data in exports.items():
    path = OUTPUT_DIR / f"trace_st_{case_name}_processed.nc"
    data.to_dataset().to_netcdf(
        path,
        encoding={
            "trace_st_data": {
                "zlib": True,
                "complevel": 4,
                "dtype": "float32",
            }
        },
    )
    manifest["files"][path.name] = {
        "sha256": sha256(path), "bytes": path.stat().st_size,
        "dimensions": {k: int(v) for k, v in data.sizes.items()},
        "variables": list(map(str, data["var"].values)),
    }

manifest_path = OUTPUT_DIR / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
print(json.dumps(manifest, indent=2))

{
  "format": "analysis-ready TraCE-ST inputs",
  "longitude_convention": "0 to 360 degrees east",
  "files": {
    "trace_st_debby_processed.nc": {
      "sha256": "50308507a15f5a25d732315fb3b51ae2eab51fe237060d1b06f559b69d7c9ab5",
      "bytes": 197274437,
      "dimensions": {
        "time": 361,
        "var": 3,
        "lat": 161,
        "lon": 361
      },
      "variables": [
        "Tb",
        "vo",
        "precip"
      ]
    },
    "trace_st_pinatubo_processed.nc": {
      "sha256": "9063424a7edf47a4e231aa6ad08660aedf6551d0e0a5202ba5ca7fb4fab722a9",
      "bytes": 504458,
      "dimensions": {
        "time": 20,
        "var": 4,
        "lat": 33,
        "lon": 90
      },
      "variables": [
        "TMSO201",
        "BURDENSO401",
        "TMH2SO401",
        "AEROD_v"
      ]
    },
    "trace_st_pnw21_processed.nc": {
      "sha256": "ba81c7b7cfa31fc8cb9cedfa13d5a5d94b3ba95e6f5db750397fac3a55d4085a",
      "bytes": 2891460,
      "dimensions": {
        "time"